In [19]:

!pip install gymnasium stable-baselines3 numpy

In [20]:
PASS_LIST = [
    "mem2reg",
    "instcombine",
    "gvn",
    "licm",
    "simplifycfg"
]

In [21]:
import subprocess

def compile_to_bc(c_file, bc_file):
    subprocess.run(
        ["clang", "-O0", "-emit-llvm", "-c", c_file, "-o", bc_file],
        check=True
    )

In [22]:
def apply_pass(input_bc, output_bc, opt_pass):
    subprocess.run(
        ["opt", f"-{opt_pass}", input_bc, "-o", output_bc],
        check=True
    )


In [23]:
def bc_to_ir(bc_file, ir_file):
    subprocess.run(
        [r"C:\Program Files\LLVM\bin\llvm-dis.exe", bc_file, "-o", ir_file],
        check=True
    )


In [ ]:
def extract_features(ir_file):
    with open(ir_file, "r") as f:
        lines = f.readlines()

    instr = 0
    blocks = 0
    funcs = 0

    for l in lines:
        l = l.strip()

        if l.startswith("%"):
            instr += 1

        if l.endswith(":"):
            blocks += 1

        if l.startswith("define"):
            funcs += 1

    return [instr, blocks, funcs, len(lines)]

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class LLVMOptEnv(gym.Env):

    def __init__(self, c_file):
        super().__init__()

        self.c_file = c_file
        self.step_id = 0

        self.action_space = spaces.Discrete(len(PASS_LIST))

        self.observation_space = spaces.Box(
            low=0,
            high=1e6,
            shape=(4,),
            dtype=np.int32
        )

        compile_to_bc(self.c_file, "current.bc")
        bc_to_ir("current.bc", "current.ll")

        self.state = np.array(extract_features("current.ll"))

    def reset(self, seed=None, options=None):

        self.step_id = 0

        compile_to_bc(self.c_file, "current.bc")
        bc_to_ir("current.bc", "current.ll")

        self.state = np.array(extract_features("current.ll"))

        return self.state, {}

    def step(self, action):

        opt_pass = PASS_LIST[action]

        new_bc = f"step_{self.step_id}.bc"

        apply_pass("current.bc", new_bc, opt_pass)

        os.replace(new_bc, "current.bc")

        bc_to_ir("current.bc", "current.ll")

        new_state = np.array(extract_features("current.ll"))

        reward = int(self.state[0] - new_state[0])

        self.state = new_state
        self.step_id += 1

        done = self.step_id >= 5

        return self.state, reward, done, False, {}

In [ ]:
env = LLVMOptEnv("test.c")

for episode in range(3):

    state, _ = env.reset()

    total_reward = 0

    for step in range(5):

        action = env.action_space.sample()

        state, reward, done, _, _ = env.step(action)

        total_reward += reward

        if done:
            break

    print(f"Episode {episode}: Total Reward = {total_reward}")

Episode 0: Total Reward = 3
Episode 1: Total Reward = 26
Episode 2: Total Reward = 39
Episode 3: Total Reward = 27
Episode 4: Total Reward = 18


In [1]:
!which clang
!which opt
!which llvm-dis

'which' is not recognized as an internal or external command,
operable program or batch file.
'which' is not recognized as an internal or external command,
operable program or batch file.
'which' is not recognized as an internal or external command,
operable program or batch file.
